In [ ]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

#username = "aacuser"
#password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = Dash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Logo and ID header
    html.Div([
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '120px', 'marginRight': '12px'}
            ),
            href='https://www.snhu.edu', target='_blank', rel='noopener noreferrer'
        ),
        html.Div([
            html.B("Grazioso Salvare Dashboard"),
            html.Div("Built by Caleb Luplow - Project Two", style={'opacity': 0.75})
        ], style={'display': 'inline-block', 'verticalAlign': 'middle', 'marginLeft': '8px'})
    ], style={'display': 'flex', 'alignItems': 'center'}),
    #html.Div(id='hidden-div', style={'display':'none'}),
    #html.Center(html.B(html.H1('Animal Shelter Dashboard'))),
    
    html.Hr(),
    
    # Interactive filter
    html.Div([
        html.Label("Rescue Type", style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset (All)', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'marginRight': '18px'}
        )
    ]),
        
    html.Hr(),
    
    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={
            'minWidth': '120px', 'width': '120px', 'maxWidth': '240px',
            'whiteSpace': 'normal'
        },
        style_header={'fontWeight': 'bold'}
    ),
    
    html.Br(),
    html.Hr(),
    
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
        style={'display' : 'flex'}, children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Filters
    if not filter_type or filter_type == 'reset':
        query = {}
    elif filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "$or": [
                {"breed": {"$regex": r"Labrador Retriever", "$options": "i"}},
                {"breed": "Chesapeake Bay Retriever"},
                {"breed": "Newfoundland"}
            ]}
    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]}
        }
    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]}
        }
    else:
        query = {}
        
    # Query MongoDB through the CRUD module
    records = db.read(query)
    
    # Normalize to DataFrame and drop _id for Dash
    dff = pd.DataFrame.from_records(records)
    if dff.empty:
        return[]
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
        
    return dff.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data to display.")]
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Use top 15 breed distribution
    breed_col = 'breed' if 'breed' in dff.columns else dff.columns[0]
    counts = dff[breed_col].value_counts().reset_index()
    counts.columns = [breed_col, 'count']
    counts = counts.head(15)
    
    fig = px.bar(
        counts,
        x='count',
        y=breed_col,
        orientation='h',
        title='Breed Distribution (Top 15 in Current View)'
    )
    fig.update_layout(margin=dict(l=10, r=10, t=40, b=10), height=460)
    
    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data available")]
    
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Because we only allow single row selection, the list can be converted to a row index here
    if not index:
        row = 0
    else: 
        row = index[0]
        
    lat = dff.iloc[row]["location_lat"]
    lon = dff.iloc[row]["location_long"]
    
    breed = dff.iloc[row]["breed"]
    name = dff.iloc[row]["name"]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(str(breed)),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(str(name))
                ])
            ])
        ])
    ]



app.run(debug=True)
print("Open http://localhost:8050")


Open http://localhost:8050


['', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks', 'name']
['', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks', 'name']
['', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks', 'name']
['', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color', 'date_of_birth', 'datetime', 'monthyear', 'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long', 'age_upon_outcome_in_weeks', 'name']
['', 'age_upon_outcome', 'animal_id', 'animal_ty